# Module 10 — Test the business rule

Write an expected answer. Run pytest. Read the failure. Repair the code.

You may use Codex, but you own the business rule and expected result. These are prepared examples of plausible mistakes, not recorded AI outputs.

**Session 1:** shipping, hotel refunds, promotions. **Session 2:** conversion, retention, and the wildfire project.

Run cells in order. Each activity creates Python files in `pytest_business_lab/`. Open those files to edit your tests and implementation.

## Setup

Select the repository’s `.venv` Python kernel in VS Code. Run `uv sync --extra notebook` in the terminal first. Starter cells preserve existing files. To restart an activity, rename its folder before rerunning it.

Response cells are collapsed for now; expand them when needed.

In [ ]:
import pytest
import pandas as pd
print("pytest", pytest.__version__, "| pandas", pd.__version__)

In [ ]:
import sys
import subprocess
import shutil
from pathlib import Path

ROOT = Path.cwd() / "pytest_business_lab"
ROOT.mkdir(exist_ok=True)

def seed(path, content):
    # Rerunning a starter cell must not erase your edits.
    if not path.exists():
        path.write_text(content, encoding="utf-8")
    else:
        print(f"Keeping existing {path.name}")

def run_tests(folder):
    work = ROOT / folder
    # Repairs can happen within one second; discard stale bytecode.
    for cache in work.rglob("__pycache__"):
        shutil.rmtree(cache)
    result = subprocess.run(
        [sys.executable, "-m", "pytest", "-q", str(work)],
        capture_output=True, text=True,
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    print("Exit code:", result.returncode)
    print("0: passed | 1: failed | 2–4: execution/setup problem | 5: no tests collected")
    return result.returncode

print("Activity files:", ROOT)


## 1. The Green Test Trap — free shipping
**Business policy (agreed by the team):** For a regular US checkout, the shipping fee is $5 when the **post-discount merchandise subtotal** is less than $50; it is $0 at $50 or above. The subtotal is already validated and nonnegative upstream. Taxes are excluded.

**Business discussion:** A cart of $55 with a $10 coupon is only $45 after discount. Should shipping be $0 or $5? Here the agreed answer is **$5**. Negative totals are not a priority test for this function because they are outside its contract.

**pytest:** Tests are ordinary functions named `test_*`; `assert` compares actual and expected behavior. A green result only covers the examples tested.

In [ ]:
folder = ROOT / "01_shipping"
folder.mkdir(exist_ok=True)
seed(folder / "shipping.py", """def shipping_fee(post_discount_subtotal):
    return 0 if post_discount_subtotal > 50 else 5
""")
seed(folder / "test_shipping.py", """from shipping import shipping_fee

def test_common_orders():
    assert shipping_fee(20) == 5
    assert shipping_fee(100) == 0
""")
run_tests("01_shipping")

**Task:** Predict whether the existing tests pass. Add a boundary test for exactly $50 and a test for $49.99. Notice that the first green result did **not** establish correctness.

`@pytest.mark.parametrize` is a concise way to run one test on several input/output pairs.

In [ ]:
# STUDENT TODO: Replace the contents of test_shipping.py with a parametrized test.
# Suggested pairs: (20, 5), (49.99, 5), (50, 0), (100, 0).
# You can ask Codex: "Write a pytest parametrized test for these agreed pairs."
print((folder / "test_shipping.py").read_text())
# After editing the file, run:
# run_tests("01_shipping")

## 2. Break My Code — hotel cancellations (pairs)
**Why this is a realistic business example:** A hotel needs to calculate refunds under a published cancellation policy, with cutoff conditions that genuinely matter.

**Simplified policy (make no additional assumptions):** All prices are in USD. `hours_before_checkin` is a precomputed nonnegative number; the booking system has already normalized time zones. For **flexible** bookings, cancellation **at least 48 hours** before check-in gives a full refund; **at least 24 but less than 48 hours** gives a 50% refund; **less than 24 hours** gives $0. **Nonrefundable** bookings give $0. Refunds never exceed the prepaid amount. Only flexible and nonrefundable rates are in scope.

**Pairs:** Partner A takes the role of developer. Partner B takes the role of skeptical QA analyst. A reads the policy and writes/edits implementation (Codex allowed). B writes tests **without looking at the function** at first (Codex allowed). Exchange tests, run pytest, discuss the failure, and switch roles for a second round.

**Prompts to ask:** Should exactly 48 hours be full or half? What about exactly 24 hours and just below it? Does a $0 prepaid booking make sense? What happens to the refund for a nonrefundable booking? Which case could distinguish `>=` from `>`?

**Already ran an earlier version?** Starter cells keep your existing files. Rename `pytest_business_lab/02_hotel` before rerunning this activity to create the updated starter without deleting your work.


In [ ]:
folder = ROOT / "02_hotel"
folder.mkdir(exist_ok=True)
seed(folder / "refunds.py", """def cancellation_refund(prepaid_amount, hours_before_checkin, rate_type):
    if rate_type == "nonrefundable":
        return 0.0
    if hours_before_checkin > 48:
        return float(prepaid_amount)
    if hours_before_checkin >= 24:
        return float(prepaid_amount) * 0.5
    return 0.0
""")

In [ ]:
seed(folder / "test_refunds.py", """from refunds import cancellation_refund

def test_full_refund_well_before_cutoff():
    assert cancellation_refund(200, 72, "flexible") == 200

# Partner B: add tests at 48 and 24 hours, just below each cutoff,
# and for nonrefundable rates. You may ask Codex to help write them.
""")
run_tests("02_hotel")
print((folder / "test_refunds.py").read_text())

**Codex prompt for Partner B:** “Here is the hotel refund policy. Propose cases covering both cutoffs and nonrefundable bookings, with inputs and expected outputs, explaining which business rule each checks. Then convert only our approved cases into `pytest.mark.parametrize`. Do not read or modify the implementation.”

**Debrief:** The job is not to maximize the number of tests. It is to choose cases that separate a correct implementation from plausible mistakes.

## 3. Promotion engine — why `pytest.approx` matters
**Policy:** A retailer discounts an eligible $120 basket by 15%. Inputs are validated upstream: subtotal is nonnegative and discount rate is between 0 and 1. The discount applies only to eligible merchandise. We will test a simple helper that receives the eligible merchandise amount.

**Business sense:** Floating-point calculations can have tiny representation differences. For monetary production systems, integer cents or decimal arithmetic is often preferable; here `pytest.approx` introduces approximate comparisons for Python floats.

**Task:** Can you catch a bug where the implementation applies the discount **twice**?

In [ ]:
folder = ROOT / "03_promo"
folder.mkdir(exist_ok=True)
seed(folder / "promotion.py", """def net_price(eligible_subtotal, discount_rate):
    discounted = eligible_subtotal * (1 - discount_rate)
    return discounted * (1 - discount_rate)
""")

In [ ]:
seed(folder / "test_promotion.py", """import pytest
from promotion import net_price

def test_no_discount():
    assert net_price(100, 0) == pytest.approx(100)

# TODO: Add a test for a $120 basket and 15% discount.
# Ask Codex to help turn your own expected answer into a test.
""")
run_tests("03_promo")

## 4. The Dashboard That Lies — duplicated customers
**Business setting:** A dashboard reports **customer purchase conversion**: among distinct visitors in an experiment, what fraction made at least one purchase? Each visitor counts **once**, even if they purchased multiple times.

A join with transaction rows can make the dashboard look plausible while counting purchasers twice.

**Our expected result:** Visitors A, B, C, D; purchases from A twice and C once. Compute the conversion rate before you run a test.

`@pytest.fixture` can provide reusable synthetic data, and `pd.testing.assert_frame_equal` / `assert_series_equal` can validate data outputs.

In [ ]:
folder = ROOT / "04_dashboard"
folder.mkdir(exist_ok=True)
seed(folder / "metrics.py", """def conversion_rate(visitors, orders):
    return len(orders) / visitors["visitor_id"].nunique()
""")

In [ ]:
seed(folder / "test_metrics.py", """import pandas as pd
import pytest
from metrics import conversion_rate

@pytest.fixture
def visitors():
    return pd.DataFrame({"visitor_id": ["A", "B", "C", "D"]})

@pytest.fixture
def orders():
    return pd.DataFrame({"visitor_id": ["A", "A", "C"], "order_id": [1, 2, 3]})

# TODO: Write test_unique_visitor_conversion(visitors, orders).
""")
run_tests("04_dashboard")

**Discussion:** The bigger problem is not the code crash—it is the plausible but incorrect metric. Codex can write tests, but the analyst must choose the unit of analysis and denominator.

**Optional extension:** Does the visitor table need deduplication? What if orders contain visitors outside the experiment?

## 5. AI Agent Tests Itself — retention definition
**Business question:** What fraction of customers who purchased in January also purchased in February?

**Agreed definition:** Retention = number of distinct January purchasers with at least one February purchase ÷ number of distinct January purchasers. Repeat transactions in either month do not change the answer.

**Dataset:** January has customers A, A, B, C; February has A, A, D. Compute the retention rate before you run a test.

**Challenge:** Have Codex generate both an implementation and tests. Before you accept them, independently write the expected result on paper. Check whether the proposed tests would catch a denominator or duplicate-counting error.

In [ ]:
folder = ROOT / "05_retention"
folder.mkdir(exist_ok=True)
seed(folder / "retention.py", """def retained_fraction(january_customers, february_customers):
    return len(set(january_customers) & set(february_customers)) / len(january_customers)
""")

In [ ]:
seed(folder / "test_retention.py", """import pytest
from retention import retained_fraction

def test_no_one_returns():
    assert retained_fraction(["A", "B"], ["C"]) == pytest.approx(0)

# TODO: Add a case with duplicate customer transactions.
# Write the expected result yourself BEFORE asking Codex for pytest syntax.
""")
run_tests("05_retention")

### Wrap-up: read tests, don’t just run them

**Exit ticket (individual, 2 minutes):**
1. Give an example of a test that passes but fails to prove business correctness.
2. In the dashboard case, what was the *unit of analysis*?
3. What are you comfortable delegating to Codex, and what must remain your decision?

**Optional Codex prompt for any exercise:** “Read the business specification and list ambiguous requirements before writing code. Propose test cases with explicit expected results, explain the business rationale for each, and wait for me to approve them. Then implement pytest tests. Never weaken an assertion solely to make the suite pass.”

## Apply this to your project

Use the wildfire dashboard and its drought files as a worked project example. Inspect `tests/test_dashboard.py` and `tests/test_drought.py`. Which business rules do they check? What is still untested?

**Homework:** write pytest tests for your full team project on your own branch. Cover its important behavior, not only your Module 9 audit. Commit the tests and `test-report.md`; explain the results and gaps. Submit the report to Brightspace.